In [1]:
import os
os.environ.pop("PYTHONPATH", None)

import sys
sys.path = [
    p for p in sys.path
    if "/opt/software-current/" not in p
]

import regex
print("regex:", regex.__file__)

regex: /mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/regex/__init__.py


In [4]:
import pandas as pd

df = pd.read_parquet("../clip_train.parquet")

In [16]:
import torch
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch.nn.functional as F

MODEL_NAME = "google/siglip2-base-patch16-224"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 16092.50it/s]


In [ ]:
def embed_images_batch(image_paths, batch_size: int = 128):
    embeddings = []


    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]

        images = []
        for path in batch_paths:
            with Image.open(path) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.inference_mode():
            output = model.get_image_features(**inputs)
            embedding = output.pooler_output
            embedding = F.normalize(embedding, p=2, dim=-1)

        embeddings.append(embedding)

    # return embedding.squeeze(0)
    return torch.cat(embeddings, dim=0)

def embed_text_batch(texts, batch_size: int = 128):
    embeddings = []

    for start in range(0, len(texts), batch_size):

        inputs = processor(text=texts[start:start + batch_size],
            padding="max_length", return_tensors="pt").to(device)

        with torch.inference_mode():
            output = model.get_text_features(**inputs)
            embedding = output.pooler_output
            embedding = F.normalize(embedding, p=2, dim=-1)

        embeddings.append(embedding)

    return torch.cat(embeddings, dim=0)

def embed_batch(df, batch_size=256, image_weight=0.6):
    image_paths = df["image_path"].astype(str).tolist()
    image_embeddings = embed_images_batch(image_paths, batch_size)

    caption_paths = df["caption"].astype(str).tolist()
    caption_embeddings = embed_text_batch(caption_paths, batch_size)

    embeddings = (image_embeddings * image_weight) + (caption_embeddings * (1 i image_weight))

    return embeddings

In [47]:
z = torch.tensor([1, 2])
x = torch.tensor([1, 2])
z /2+ x

tensor([1.5000, 3.0000])

In [36]:
image_paths = df["image_path"].astype(str).tolist()

In [37]:
z = embed_batch(image_paths[:200])
z.shape

torch.Size([200, 768])

In [44]:
captions = df["caption"].astype(str).tolist()
z = embed_text_batch(captions[:200])
z.shape

torch.Size([200, 768])

In [9]:
image_embeddings = torch.stack([
    embed(str(row["image_path"])).squeeze(0).cpu()
    for _, row in df[:10].iterrows()
]).cpu().numpy()

# caption_embeddings = torch.stack([
#     embed_text(str(caption)).cpu()
#     for caption in df["caption"]
# ]).cpu().numpy()

In [28]:
import torch

edge_index = torch.randint(0, 25000, (2, 100000)).cuda()
features = torch.randn(25000, 512).cuda()
features

tensor([[ 0.0693,  0.0524, -1.4087,  ..., -0.0190, -0.3557, -1.4801],
        [ 1.3131, -1.4798,  0.3522,  ...,  0.6230, -0.4607, -0.4374],
        [-0.5870, -0.0303, -1.7523,  ...,  0.9206, -0.2960, -0.6254],
        ...,
        [-0.1415, -0.7528, -0.5821,  ..., -0.8134, -0.1963, -0.6935],
        [-0.7804,  0.1427,  1.8880,  ...,  1.3926,  0.4718, -0.1688],
        [-0.2184,  0.0114,  0.3858,  ..., -0.7372, -0.2114, -0.6227]],
       device='cuda:0')

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [26]:
class GraphSAGE(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, output_dim=128):
        super().__init__()

        self.layer1 = SAGEConv(512, 256)
        self.layer2 = SAGEConv(256, 128)

    def forward(self, x, edge_index):
        x = F.relu(self.layer1(x, edge_index))
        x = self.layer2(x, edge_index)

        return x

model = GraphSAGE().cuda()

In [ ]:
out = model(features, edge_index)

: 